<a href="https://colab.research.google.com/github/jiffy-jacob/Gen_AI_Paraphrasing-tool/blob/main/gen_ai_paraphrase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#AI-powered paraphrasing tool
Develop an AI-powered paraphrasing tool as a Python application or module. The tool
should:

● Take a block of input text

● Use deep learning (preferably transformer models like T5, BERT, GPT, or similar
via Hugging Face Transformers)

● Generate a paraphrased version preserving meaning, improving clarity, and
ensuring originality.

● Include built-in checks for grammar, spelling, and fluency of output

Recommended technologies:

● Python, with Hugging Face Transformers and relevant NLP packages (spaCy,
NLTK, etc.)

● TensorFlow or PyTorch for any model fine-tuning

● Use only console or script-based input/output (no GUI or web interface)

● Optional: Integrate evaluation metrics like BLEU, ROUGE, or semantic similarity
scores

In [1]:
#Install Dependencies
%pip install -q transformers torch sentencepiece language-tool-python nltk rouge-score sentence-transformers scikit-learn pandas


In [2]:
#Import packages and select the Humarin model
#The checkpoint is a T5-base paraphrasing model. We use its sequence-to-sequence interface.
import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "humarin/chatgpt_paraphraser_on_T5_base"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


Using device: cpu


In [3]:
#Load tokenizer and model
#This downloads model files on the first run and caches them for later runs.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("Loaded:", MODEL_NAME)


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Loaded: humarin/chatgpt_paraphraser_on_T5_base


In [4]:
#Split long input into manageable chunks
#The model card's example uses a maximum sequence length of 128. We split long
# input so that it is not silently truncated.
MAX_INPUT_TOKENS = 120

def split_long_sentence(sentence, token_limit=MAX_INPUT_TOKENS):
    words = sentence.split()
    pieces, current = [], []
    for word in words:
        candidate = " ".join(current + [word])
        size = len(tokenizer.encode(candidate, add_special_tokens=True))
        if current and size > token_limit:
            pieces.append(" ".join(current))
            current = [word]
        else:
            current.append(word)
    if current:
        pieces.append(" ".join(current))
    return pieces

def make_chunks(text, token_limit=MAX_INPUT_TOKENS):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current = [], ""
    for sentence in sentences:
        if not sentence.strip():
            continue
        for piece in split_long_sentence(sentence, token_limit):
            candidate = f"{current} {piece}".strip()
            if current and len(tokenizer.encode(candidate, add_special_tokens=True)) > token_limit:
                chunks.append(current)
                current = piece
            else:
                current = candidate
    if current:
        chunks.append(current)
    return chunks

print("Chunking helpers are ready.")


Chunking helpers are ready.


In [5]:
#definition of paraphrase function
def paraphrase(text):
    if not text or not text.strip():
        raise ValueError("Please provide non-empty input text.")
    results = []
    for chunk in make_chunks(text):
        inputs = tokenizer(
            f"paraphrase: {chunk}",
            return_tensors="pt",
            truncation=True,
            max_length=128,
        ).to(device)
        with torch.inference_mode():
            generated = model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=5,
                num_beam_groups=5,
                num_return_sequences=1,
                repetition_penalty=10.0,
                diversity_penalty=3.0,
                no_repeat_ngram_size=2,
                trust_remote_code=True, # This is the fix
                custom_generate='transformers-community/group-beam-search' # Additional fix
            )
        results.append(tokenizer.decode(generated[0], skip_special_tokens=True).strip())
    return " ".join(results)

#testing
original_text = (
    "Regular exercise can improve physical health and reduce stress. "
    "Even a short daily walk may help people feel more energetic."
)
paraphrased_text = paraphrase(original_text)
print("ORIGINAL\n", original_text)
print("\nPARAPHRASE\n", paraphrased_text)


ORIGINAL
 Regular exercise can improve physical health and reduce stress. Even a short daily walk may help people feel more energetic.

PARAPHRASE
 Regular exercise can promote physical health and alleviate stress. A short walk a day may boost energy levels in some people.


In [6]:
#Check grammar and spelling
import language_tool_python

def grammar_spelling_suggestions(text):
    tool = language_tool_python.LanguageTool("en-US")
    try:
        matches = tool.check(text)
        return [
            {
                "rule": match.ruleId,
                "message": match.message,
                "position": match.offset,
                "suggestions": match.replacements[:3],
            }
            for match in matches
        ]
    finally:
        tool.close()

suggestions = grammar_spelling_suggestions(paraphrased_text)
if suggestions:
    for suggestion in suggestions:
        print(suggestion)
else:
    print("No issues flagged. Review the text manually too.")


No issues flagged. Review the text manually too.


##Evaluate the text file and print console-style output

The program reads evaluation_data.txt. Each line can be original textreference paraphrase, or just original text. For every generated output, LanguageTool checks grammar, spelling, and style; its flags per 100 words are shown as a rough fluency indicator. Metrics and check results are printed and saved to evaluation_results.csv.

In [7]:
from pathlib import Path
import csv
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import language_tool_python

INPUT_FILE = Path("evaluation_data.txt")
OUTPUT_FILE = Path("evaluation_results.csv")

# Start: Ensure INPUT_FILE exists for demonstration/execution
if not INPUT_FILE.exists():
    print(f"Creating a dummy '{INPUT_FILE}' for demonstration purposes.")
    dummy_content = """# This is a dummy evaluation data file.\n# Each line should contain an original text, optionally followed by a tab and a reference paraphrase.\nRegular exercise can improve physical health and reduce stress.\tRegular physical activity boosts health and reduces tension.\nEven a short daily walk may help people feel more energetic.\tA brief daily stroll can increase energy levels.\nThis is a test sentence without a reference.\n"""
    INPUT_FILE.write_text(dummy_content, encoding="utf-8")
# End

def read_evaluation_file(file_path):
    records = []
    for line_number, raw_line in enumerate(file_path.read_text(encoding="utf-8-sig").splitlines(), start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        fields = line.split("\t", maxsplit=1)
        source = fields[0].strip()
        reference = fields[1].strip() if len(fields) == 2 else ""
        if not source:
            raise ValueError(f"Empty source text on line {line_number}.")
        records.append({"source": source, "reference": reference})
    if not records:
        raise ValueError(f"No input records found in {file_path.resolve()}.")
    return records

records = read_evaluation_file(INPUT_FILE)
print("=" * 72)
print("AI-POWERED PARAPHRASING TOOL")
print("=" * 72)
print(f"Input file : {INPUT_FILE.resolve()}")
print(f"Records    : {len(records)}")

similarity_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
quality_tool = language_tool_python.LanguageTool("en-US")
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method1
results = []

for index, record in enumerate(records, start=1):
    generated = paraphrase(record["source"])
    matches = quality_tool.check(generated)
    issue_rate = len(matches) * 100 / max(len(generated.split()), 1)
    vectors = similarity_model.encode([record["source"], generated], convert_to_numpy=True)
    semantic_score = float(cosine_similarity([vectors[0]], [vectors[1]])[0][0])
    result = {
        "id": index,
        "original": record["source"],
        "reference": record["reference"],
        "paraphrase": generated,
        "grammar_spelling_style_flags": len(matches),
        "language_flags_per_100_words": issue_rate,
        "semantic_similarity": semantic_score,
        "bleu": None,
        "rouge1_f1": None,
        "rouge2_f1": None,
        "rougeL_f1": None,
    }
    if record["reference"]:
        reference_tokens = record["reference"].lower().split()
        generated_tokens = generated.lower().split()
        result["bleu"] = sentence_bleu(
            [reference_tokens], generated_tokens, smoothing_function=smooth
        )
        rouge_scores = rouge.score(record["reference"], generated)
        result["rouge1_f1"] = rouge_scores["rouge1"].fmeasure
        result["rouge2_f1"] = rouge_scores["rouge2"].fmeasure
        result["rougeL_f1"] = rouge_scores["rougeL"].fmeasure
    results.append(result)

    print()
    print(f"[{index}/{len(records)}] RESULT")
    print("-" * 72)
    print("INPUT:")
    print(record["source"])
    if record["reference"]:
        print("\nREFERENCE:")
        print(record["reference"])
    print("\nPARAPHRASE:")
    print(generated)
    print("\nGRAMMAR / SPELLING / FLUENCY CHECK:")
    print(f"  LanguageTool flags    : {len(matches)}")
    print(f"  Flags per 100 words   : {issue_rate:.2f} (rough fluency indicator)")
    for match in matches[:5]:
        fixes = ", ".join(match.replacements[:3]) or "no replacement suggested"
        print(f"  - {match.message} | suggestions: {fixes}")
    if not matches:
        print("  No issues flagged.")
    print("\nMETRICS:")
    print(f"  Semantic similarity : {semantic_score:.4f}")
    if record["reference"]:
        print(f"  BLEU                : {result['bleu']:.4f}")
        print(f"  ROUGE-1 F1          : {result['rouge1_f1']:.4f}")
        print(f"  ROUGE-2 F1          : {result['rouge2_f1']:.4f}")
        print(f"  ROUGE-L F1          : {result['rougeL_f1']:.4f}")
    else:
        print("  BLEU / ROUGE        : skipped (no reference paraphrase)")
    print("-" * 72)

quality_tool.close()
with OUTPUT_FILE.open("w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)
print()
print("Evaluation complete.")
print(f"Detailed results saved to: {OUTPUT_FILE.resolve()}")


AI-POWERED PARAPHRASING TOOL
Input file : /content/evaluation_data.txt
Records    : 3


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[1/3] RESULT
------------------------------------------------------------------------
INPUT:
Regular exercise can improve physical health and reduce stress.

REFERENCE:
Regular physical activity boosts health and reduces tension.

PARAPHRASE:
Exercise is beneficial for both physical health and stress relief.

GRAMMAR / SPELLING / FLUENCY CHECK:
  LanguageTool flags    : 0
  Flags per 100 words   : 0.00 (rough fluency indicator)
  No issues flagged.

METRICS:
  Semantic similarity : 0.8784
  BLEU                : 0.0494
  ROUGE-1 F1          : 0.3333
  ROUGE-2 F1          : 0.1250
  ROUGE-L F1          : 0.3333
------------------------------------------------------------------------

[2/3] RESULT
------------------------------------------------------------------------
INPUT:
Even a short daily walk may help people feel more energetic.

REFERENCE:
A brief daily stroll can increase energy levels.

PARAPHRASE:
Those who walk for a few minutes each day may experience an increase in energy 

In [8]:
#Review console summary
def mean_of(metric):
    values = [row[metric] for row in results if row[metric] is not None]
    return sum(values) / len(values) if values else None

print()
print("=" * 72)
print("SUMMARY")
print("=" * 72)
for metric, label in [
    ("semantic_similarity", "Mean semantic similarity"),
    ("language_flags_per_100_words", "Mean flags per 100 words"),
    ("bleu", "Mean BLEU"),
    ("rouge1_f1", "Mean ROUGE-1 F1"),
    ("rouge2_f1", "Mean ROUGE-2 F1"),
    ("rougeL_f1", "Mean ROUGE-L F1"),
]:
    value = mean_of(metric)
    print(f"{label:30}: {value:.4f}" if value is not None else f"{label:30}: N/A (references required)")
print("=" * 72)




SUMMARY
Mean semantic similarity      : 0.7453
Mean flags per 100 words      : 0.0000
Mean BLEU                     : 0.0402
Mean ROUGE-1 F1               : 0.3333
Mean ROUGE-2 F1               : 0.1080
Mean ROUGE-L F1               : 0.3333


##Conclusion
The paraphrasing system successfully generated alternative versions of human-written texts using the humarin/chatgpt_paraphraser_on_T5_base model. The generated sentences were evaluated against the human reference texts using ROUGE metrics to measure textual similarity. The results help determine how much the AI-generated text differs from the original while maintaining the same information. Since ROUGE mainly measures word and phrase overlap, semantic similarity evaluation can provide additional insight into whether the generated paraphrases preserve the original meaning. Overall, this project demonstrates how transformer-based models can be used for automatic text paraphrasing and evaluated using both lexical and semantic measures.